## imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# regressors
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

# classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")

## 1. data audit

In [ ]:
# search kaggle path dynamically
data_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".csv") and "coaster" in file.lower():
            data_path = os.path.join(root, file)
            break

if data_path is None:
    data_path = "coaster_db.csv"

df = pd.read_csv(data_path)
print(f"loaded dataset from: {data_path}")

# standardizing column names
col_map = {col.lower(): col for col in df.columns}
speed_col = col_map.get('speed_mph', 'speed_mph')
height_col = col_map.get('height_ft', 'height_ft')
length_col = col_map.get('length_ft', 'Length_ft' if 'Length_ft' in df.columns else 'length_ft')
inversions_col = col_map.get('inversions_clean', 'Inversions_clean' if 'Inversions_clean' in df.columns else 'inversions_clean')
gforce_col = col_map.get('gforce_clean', 'Gforce_clean' if 'Gforce_clean' in df.columns else 'gforce_clean')
type_col = col_map.get('type_main', 'Type_Main' if 'Type_Main' in df.columns else 'type_main')

print("\n--- shape ---")
print(df.shape)

print("\n--- dtypes ---")
print(df.dtypes)

print("\n--- missing values ---")
print(df.isnull().sum())

print("\n--- duplicates ---")
print(df.duplicated().sum())

print(f"\n--- target stats ({speed_col}) ---")
print(df[speed_col].describe())

## 2. EDA

In [ ]:
# target distribution
plt.figure(figsize=(8, 4))
sns.histplot(df[speed_col].dropna(), kde=True, color="teal")
plt.title(f"target distribution: {speed_col}")
plt.xlabel("speed (mph)")
plt.ylabel("count")
plt.tight_layout()
plt.show()

**target check:** speed is pretty normal looking around 48 mph. no transformation needed.

In [ ]:
# numerical distributions
num_cols_eda = [c for c in [speed_col, height_col, length_col, inversions_col, gforce_col] if c in df.columns]
df[num_cols_eda].hist(figsize=(12, 8), bins=20, color="skyblue", edgecolor="black")
plt.suptitle("numerical features", fontsize=14)
plt.tight_layout()
plt.show()

**feature check:** height and length are right skewed from a few really big coasters. inversions is mostly zeros. will clean up extreme outliers before training.

In [ ]:
# correlation matrix
plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols_eda].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("correlation matrix")
plt.tight_layout()
plt.show()

**correlations:** height and length correlate with speed and each other, so linear models might see collinearity. ridge and lasso should help.

In [ ]:
# scatter plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if height_col in df.columns:
    sns.scatterplot(data=df, x=height_col, y=speed_col, ax=axes[0], color="purple")
    axes[0].set_title(f"{speed_col} vs {height_col}")
if length_col in df.columns:
    sns.scatterplot(data=df, x=length_col, y=speed_col, ax=axes[1], color="orange")
    axes[1].set_title(f"{speed_col} vs {length_col}")
plt.tight_layout()
plt.show()

**scatter plots:** positive trend overall. variance gets wider at higher values so non linear models should perform well here.

Target speed looks normal around 48 mph. Height and length correlate with speed and with each other, and feature skew justifies both outlier checks and non linear models.

## 3. preprocessing

In [ ]:
# drop rows without target
df_clean = df.dropna(subset=[speed_col]).copy()

# drop duplicate coasters
dup_cols = [c for c in ["Coaster_Name", "Location"] if c in df_clean.columns]
if dup_cols:
    before = len(df_clean)
    df_clean = df_clean.drop_duplicates(subset=dup_cols).reset_index(drop=True)
    print(f"dropped {before - len(df_clean)} duplicate coaster rows")

**outliers:** using iqr to flag bad rows. only dropping if a row is an outlier on multiple features so we keep legitimate records.

In [ ]:
# outlier check (IQR rule) on key numeric columns
outlier_check_cols = [c for c in [speed_col, height_col, length_col] if c in df_clean.columns]
outlier_flags = pd.DataFrame(index=df_clean.index)

for c in outlier_check_cols:
    q1, q3 = df_clean[c].quantile(0.25), df_clean[c].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_flags[c] = (df_clean[c] < lower) | (df_clean[c] > upper)
    print(f"{c}: {outlier_flags[c].sum()} rows outside [{lower:.1f}, {upper:.1f}]")

# flag rows that are outliers on 2+ of the checked columns as likely data-entry issues
multi_outlier_mask = outlier_flags.sum(axis=1) >= 2
print(f"\nrows flagged as outliers on 2+ features (dropped): {multi_outlier_mask.sum()}")
df_clean = df_clean.loc[~multi_outlier_mask].reset_index(drop=True)

In [ ]:
# steepness feature
if height_col in df_clean.columns and length_col in df_clean.columns:
    df_clean["height_to_length_ratio"] = df_clean[height_col] / (df_clean[length_col] + 1)
    ratio_col = "height_to_length_ratio"
else:
    ratio_col = None

**new feature:** adding height to length ratio to capture steepness. gives tree models extra signal beyond raw dimensions.

In [ ]:
num_features = [c for c in [height_col, length_col, inversions_col, ratio_col] if c and c in df_clean.columns]
cat_features = [c for c in [type_col] if c in df_clean.columns]

X = df_clean[num_features + cat_features]
y = df_clean[speed_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

def get_preprocessor():
    transformers = []
    if num_features:
        num_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])
        transformers.append(("num", num_transformer, num_features))
    if cat_features:
        cat_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ])
        transformers.append(("cat", cat_transformer, cat_features))
    return ColumnTransformer(transformers=transformers)

print(f"train shape: {X_train.shape}, test shape: {X_test.shape}")

Everything is inside a Pipeline so transformation fits only on train data to prevent leakage.

## 4. regression

Running all 10 regressors: Linear, Ridge, Lasso, ElasticNet, Poly, Decision Tree, Random Forest, Gradient Boosting, SVR, and KNN.

In [ ]:
# Polynomial Regression is PolynomialFeatures -> LinearRegression, degree 2 by default
poly_reg = Pipeline(steps=[
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("lin", LinearRegression())
])

regressors = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(random_state=SEED),
    "Lasso": Lasso(random_state=SEED),
    "ElasticNet": ElasticNet(random_state=SEED),
    "Polynomial Regression (deg=2)": poly_reg,
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=SEED),
    "Random Forest Regressor": RandomForestRegressor(random_state=SEED),
    "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=SEED),
    "SVR": SVR(),
    "KNN Regressor": KNeighborsRegressor()
}

results = []
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

for name, reg in regressors.items():
    pipe = Pipeline(steps=[("prep", get_preprocessor()), ("reg", reg)])

    cv_scores = cross_val_score(pipe, X_train, y_train, cv=kf, scoring="r2")
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    results.append({
        "Model": name,
        "Mean CV R2": cv_scores.mean(),
        "Test R2": r2_score(y_test, preds),
        "RMSE": np.sqrt(mean_squared_error(y_test, preds)),
        "MAE": mean_absolute_error(y_test, preds)
    })

results_df = pd.DataFrame(results).sort_values(by="Test R2", ascending=False)
print(results_df.to_string(index=False))

Tuning the top two models: Random Forest and Gradient Boosting.

In [ ]:
# hyperparameter tuning -- model 1: Random Forest Regressor
rf_pipe = Pipeline(steps=[("prep", get_preprocessor()), ("reg", RandomForestRegressor(random_state=SEED))])

rf_param_grid = {
    "reg__n_estimators": [50, 100, 200],
    "reg__max_depth": [None, 10, 20],
    "reg__min_samples_split": [2, 5]
}

grid_rf = GridSearchCV(rf_pipe, rf_param_grid, cv=kf, scoring="r2", n_jobs=-1)
grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_
tuned_rf_preds = best_rf.predict(X_test)

baseline_rf_r2 = results_df.loc[results_df["Model"] == "Random Forest Regressor", "Test R2"].values[0]
tuned_rf_r2 = r2_score(y_test, tuned_rf_preds)

print("Random Forest -- best params:", grid_rf.best_params_)
print(f"baseline test r2: {baseline_rf_r2:.4f}  ->  tuned test r2: {tuned_rf_r2:.4f}  "
      f"(delta: {tuned_rf_r2 - baseline_rf_r2:+.4f})")
print(f"tuned test rmse: {np.sqrt(mean_squared_error(y_test, tuned_rf_preds)):.4f}")

In [ ]:
# hyperparameter tuning -- model 2: Gradient Boosting Regressor
gb_pipe = Pipeline(steps=[("prep", get_preprocessor()), ("reg", GradientBoostingRegressor(random_state=SEED))])

gb_param_grid = {
    "reg__n_estimators": [100, 200],
    "reg__learning_rate": [0.05, 0.1, 0.2],
    "reg__max_depth": [2, 3, 4]
}

grid_gb = GridSearchCV(gb_pipe, gb_param_grid, cv=kf, scoring="r2", n_jobs=-1)
grid_gb.fit(X_train, y_train)

best_gb = grid_gb.best_estimator_
tuned_gb_preds = best_gb.predict(X_test)

baseline_gb_r2 = results_df.loc[results_df["Model"] == "Gradient Boosting Regressor", "Test R2"].values[0]
tuned_gb_r2 = r2_score(y_test, tuned_gb_preds)

print("Gradient Boosting -- best params:", grid_gb.best_params_)
print(f"baseline test r2: {baseline_gb_r2:.4f}  ->  tuned test r2: {tuned_gb_r2:.4f}  "
      f"(delta: {tuned_gb_r2 - baseline_gb_r2:+.4f})")
print(f"tuned test rmse: {np.sqrt(mean_squared_error(y_test, tuned_gb_preds)):.4f}")

# pick whichever tuned model is best on test R2 for the plots below
if tuned_gb_r2 >= tuned_rf_r2:
    best_overall = best_gb
    tuned_preds = tuned_gb_preds
    best_overall_name = "Gradient Boosting Regressor (tuned)"
else:
    best_overall = best_rf
    tuned_preds = tuned_rf_preds
    best_overall_name = "Random Forest Regressor (tuned)"

print(f"\nbest overall tuned model: {best_overall_name}")

In [ ]:
# feature importances & diagnostic plots (best overall tuned tree-based model)
all_feature_names = num_features.copy()
if cat_features and "cat" in best_overall.named_steps["prep"].named_transformers_:
    cat_encoder = best_overall.named_steps["prep"].named_transformers_["cat"].named_steps["encoder"]
    encoded_cat_cols = list(cat_encoder.get_feature_names_out(cat_features))
    all_feature_names.extend(encoded_cat_cols)

importances = best_overall.named_steps["reg"].feature_importances_
feat_imp_df = pd.DataFrame({"Feature": all_feature_names, "Importance": importances}).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(data=feat_imp_df.head(10), x="Importance", y="Feature", palette="mako")
plt.title(f"feature importances ({best_overall_name})")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(y_test, tuned_preds, alpha=0.7, color="royalblue")
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2)
axes[0].set_xlabel("actual speed")
axes[0].set_ylabel("predicted speed")
axes[0].set_title(f"actual vs predicted ({best_overall_name})")

residuals = y_test - tuned_preds
axes[1].scatter(tuned_preds, residuals, alpha=0.7, color="crimson")
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set_xlabel("predicted speed")
axes[1].set_ylabel("residuals")
axes[1].set_title("residual plot")
plt.tight_layout()
plt.show()

## 5. classification track (Pokemon dataset)

Using Pokemon data for classification since predicting `is_legendary` is a natural binary task.

## 5.1 dataset audit

In [ ]:
poke_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".csv") and "pokemon" in file.lower():
            poke_path = os.path.join(root, file)
            break

if poke_path is None:
    poke_path = "pokemon.csv"

poke_df = pd.read_csv(poke_path)
print(f"loaded dataset from: {poke_path}")

print("\n--- shape ---")
print(poke_df.shape)

print("\n--- dtypes (first 20 cols) ---")
print(poke_df.dtypes.head(20))

print("\n--- missing values (nonzero only) ---")
missing = poke_df.isnull().sum()
print(missing[missing > 0])

print("\n--- duplicates ---")
print(poke_df.duplicated().sum())

print("\n--- target balance (is_legendary) ---")
print(poke_df["is_legendary"].value_counts())
print(poke_df["is_legendary"].value_counts(normalize=True))

**class balance:** target is imbalanced with few legendaries. tracking precision recall and f1 instead of raw accuracy.

## 5.2 EDA

In [ ]:
stat_cols = [c for c in ["hp", "attack", "defense", "sp_attack", "sp_defense", "speed", "base_total"]
             if c in poke_df.columns]

# class balance
plt.figure(figsize=(5, 4))
sns.countplot(data=poke_df, x="is_legendary", hue="is_legendary", palette="Set2", legend=False)
plt.title("target distribution: is_legendary")
plt.xlabel("is_legendary")
plt.ylabel("count")
plt.tight_layout()
plt.show()

# stat distributions
poke_df[stat_cols].hist(figsize=(12, 8), bins=20, color="lightseagreen", edgecolor="black")
plt.suptitle("base stat distributions", fontsize=14)
plt.tight_layout()
plt.show()

**stat distributions:** base stats skew right. legendary pokemon likely sit in the upper tail.

In [ ]:
# correlation heatmap of stats
plt.figure(figsize=(8, 6))
sns.heatmap(poke_df[stat_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("correlation matrix (base stats)")
plt.tight_layout()
plt.show()

# stat vs legendary status
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=poke_df, x="is_legendary", y="base_total", ax=axes[0], hue="is_legendary",
            palette="Set2", legend=False)
axes[0].set_title("base_total vs is_legendary")
sns.boxplot(data=poke_df, x="is_legendary", y="speed", ax=axes[1], hue="is_legendary",
            palette="Set2", legend=False)
axes[1].set_title("speed vs is_legendary")
plt.tight_layout()
plt.show()

**stat comparisons:** stats correlate with each other. base total shows a clear split between regular and legendary pokemon.

## 5.3 preprocessing

In [ ]:
poke_num_features = stat_cols + [c for c in ["height_m", "weight_kg", "capture_rate", "base_egg_steps"]
                                  if c in poke_df.columns]
poke_cat_features = [c for c in ["type1"] if c in poke_df.columns]

Xp = poke_df[poke_num_features + poke_cat_features].copy()
# capture_rate is sometimes stored as text with stray characters in this dataset -- coerce to numeric
if "capture_rate" in Xp.columns:
    Xp["capture_rate"] = pd.to_numeric(Xp["capture_rate"], errors="coerce")

yp = poke_df["is_legendary"]

Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    Xp, yp, test_size=0.2, random_state=SEED, stratify=yp
)

def get_poke_preprocessor():
    transformers = []
    if poke_num_features:
        num_t = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])
        transformers.append(("num", num_t, poke_num_features))
    if poke_cat_features:
        cat_t = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ])
        transformers.append(("cat", cat_t, poke_cat_features))
    return ColumnTransformer(transformers=transformers)

print(f"train shape: {Xp_train.shape}, test shape: {Xp_test.shape}")
print(f"train class balance:\n{yp_train.value_counts(normalize=True)}")

Preprocessing is inside a pipeline to prevent leakage. Stratified split maintains class ratios.

## 5.4 classification models

In [ ]:
poke_classifiers = {
    "Logistic Regression": LogisticRegression(random_state=SEED, max_iter=1000),
    "KNN Classifier": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Decision Tree Classifier": DecisionTreeClassifier(random_state=SEED),
    "SVC": SVC(random_state=SEED)
}

poke_cls_results = {}
skf_p = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for name, clf in poke_classifiers.items():
    pipe_cls = Pipeline(steps=[("prep", get_poke_preprocessor()), ("clf", clf)])

    cv_acc = cross_val_score(pipe_cls, Xp_train, yp_train, cv=skf_p, scoring="accuracy")
    cv_f1 = cross_val_score(pipe_cls, Xp_train, yp_train, cv=skf_p, scoring="f1")
    pipe_cls.fit(Xp_train, yp_train)
    preds_c = pipe_cls.predict(Xp_test)

    poke_cls_results[name] = {
        "pipe": pipe_cls,
        "preds": preds_c,
        "row": {
            "Model": name,
            "Mean CV Accuracy": cv_acc.mean(),
            "Mean CV F1": cv_f1.mean(),
            "Test Accuracy": accuracy_score(yp_test, preds_c),
            "Precision": precision_score(yp_test, preds_c, zero_division=0),
            "Recall": recall_score(yp_test, preds_c, zero_division=0),
            "F1-Score": f1_score(yp_test, preds_c, zero_division=0)
        }
    }

poke_cls_df = pd.DataFrame([v["row"] for v in poke_cls_results.values()]).sort_values(by="F1-Score", ascending=False)
print(poke_cls_df.to_string(index=False))

Checking confusion matrices to track false negatives.

In [ ]:
fig, axes = plt.subplots(1, len(poke_cls_results), figsize=(4 * len(poke_cls_results), 4))
if len(poke_cls_results) == 1:
    axes = [axes]

for ax, (name, info) in zip(axes, poke_cls_results.items()):
    cm = confusion_matrix(yp_test, info["preds"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Not Legendary", "Legendary"],
                yticklabels=["Not Legendary", "Legendary"], ax=ax, cbar=False)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("predicted")
    ax.set_ylabel("actual")

plt.tight_layout()
plt.show()

for name, info in poke_cls_results.items():
    print(f"\n--- classification report ({name}) ---")
    print(classification_report(yp_test, info["preds"], target_names=["Not Legendary", "Legendary"]))

**results summary:** tree and margin based models handle the threshold split well. naive bayes drops in performance because features are correlated rather than independent.